<a href="https://colab.research.google.com/github/RuiRodrigues-lab/DataScienceFE/blob/Locker/CP4_4(TH).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
import pandas as pd
from pandas.api.types import CategoricalDtype
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.stats.diagnostic import lilliefors
from scipy.stats import wilcoxon
import rpy2.robjects as ro
from scipy.stats import ttest_ind #independente
from scipy.stats import ttest_rel, t #emparelhados
from scipy.stats import mannwhitneyu

#Precisamos desta biblioteca para podermos escolher um ficheiro local
#Se o ficheiro vier por API ou tivermos um link, é so alterar a forma de import
from google.colab import files

# 1️⃣ Faz upload do ficheiro (vai abrir uma janela para escolher no teu PC)
uploaded = files.upload()

# 2️⃣ Guarda o nome do ficheiro (Colab mostra o nome depois do upload)
filename = list(uploaded.keys())[0]

# 3️⃣ Lê o Excel, por default lê sempre a primeira tab, por isso podemos usar o "sheet_name"
# Se tivermos dados em varias tabs, devemos usar uma Dataframe(df) para cada uma das tabs
df = pd.read_excel(filename, sheet_name='Exerc4')
df.head()

Saving CP4.xlsx to CP4 (2).xlsx


,Grau,Departamento
0,Nemmuitonempoucosatisfeito,Engenharia
1,Satisfeito,Engenharia
2,Poucosatisfeito,Engenharia
3,Muitosatisfeito,Marketing
4,Satisfeito,Marketing


In [18]:
# 1) Ordem correcta das categorias (como em R)
categorias_ordem = [
    "Poucosatisfeito",
    "Nemmuitonempoucosatisfeito",
    "Satisfeito",
    "Muitosatisfeito",
    "Completamentesatisfeito"
]

# 2) Converter Grau em ordinal ordenado e depois em numérico 1–5
df["Grau"] = pd.Categorical(
    df["Grau"],
    categories=categorias_ordem,
    ordered=True
)
df["Grau_num"] = df["Grau"].cat.codes +1 #enumerar as categorias

# 3) Departamento como factor com níveis Engenharia / Marketing
df["Departamento"] = pd.Categorical(
    df["Departamento"],
    categories=["Engenharia", "Marketing"]
)

print("Primeiras linhas:")
print(df[["Grau", "Grau_num", "Departamento"]].head())

# 4) Grupos em Python (Engenharia vs Marketing)
grupoE = df.loc[df["Departamento"] == "Engenharia", "Grau_num"].dropna()
grupoM = df.loc[df["Departamento"] == "Marketing", "Grau_num"].dropna()


print("\nSummary Grau_num (global):")
print("Min.   :", df["Grau_num"].min())
print("1st Qu.:", np.percentile(df["Grau_num"].dropna(), 25))
print("Median :", np.median(df["Grau_num"].dropna()))
print("Mean   :", np.mean(df["Grau_num"].dropna()))
print("3rd Qu.:", np.percentile(df["Grau_num"].dropna(), 75))
print("Max.   :", df["Grau_num"].max())

# 5) Mann–Whitney em SciPy (aproximação; p-value pode diferir do R)
stat_scipy, p_scipy = mannwhitneyu(grupoE, grupoM, alternative="less")
print("\nMann–Whitney (SciPy, approx):")
print("U =", stat_scipy)
print("p-value =", p_scipy)

Primeiras linhas:
                         Grau  Grau_num Departamento
0  Nemmuitonempoucosatisfeito         2   Engenharia
1                  Satisfeito         3   Engenharia
2             Poucosatisfeito         1   Engenharia
3             Muitosatisfeito         4    Marketing
4                  Satisfeito         3    Marketing

Summary Grau_num (global):
Min.   : 1
1st Qu.: 2.0
Median : 2.0
Mean   : 2.3666666666666667
3rd Qu.: 3.0
Max.   : 4

Mann–Whitney (SciPy, approx):
U = 32.5
p-value = 0.0003089924925946016


In [19]:
# grupos já limpos e com Grau_num = 1..5
grupoE = df.loc[df["Departamento"] == "Engenharia", "Grau_num"]
grupoM = df.loc[df["Departamento"] == "Marketing", "Grau_num"]

# H1: grau_médio_Engenharia < grau_médio_Marketing
stat, p_value = mannwhitneyu(grupoE, grupoM, alternative="less")

print("Mann–Whitney U test (Engenharia < Marketing)")
print("U =", stat)
print("p-value =", p_value)

Mann–Whitney U test (Engenharia < Marketing)
U = 32.5
p-value = 0.0003089924925946016
